# 05 — Haystack Integration: Filtered Hybrid Retrieval with SimlarDocumentStore

`SimlarDocumentStore` is a Haystack 3.x document store backed by simlar's `FilteredIndex` over a `HelixIndex`.
It comes with three retriever components — `SimlarEmbeddingRetriever`, `SimlarBM25Retriever` and `SimlarHybridRetriever` — that slot into any Haystack `Pipeline`.

Haystack `filters` are resolved in SQL to the matching documents *before* ranking, so a filtered search returns the exact top-k among the matching documents.

**What we cover**
- Embedding documents with `SentenceTransformersDocumentEmbedder` and writing them to `SimlarDocumentStore`
- A query pipeline: `SentenceTransformersTextEmbedder` → `SimlarHybridRetriever`
- Metadata filtering with the Haystack filter syntax, at run time and at init time (`filter_policy`)
- Embedding-only and BM25-only retrieval with the same filters
- Role-based access with `grant` / `roles`
- Metadata introspection, bulk updates and deletes by filter
- Saving and reloading the store, and serializing the pipeline

In [ ]:
%pip install -q -U datasets "haystack-ai>=3,<4" sentence-transformers sentence-transformers-haystack

## Load the dataset

We use [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — 120,000 news headlines and descriptions across four categories: **World**, **Sports**, **Business** and **Sci/Tech**.

We index the first 500 articles from the training split to keep the demo fast.

In [ ]:
from datasets import load_dataset

LABEL_NAMES = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

ds = load_dataset("fancyzhx/ag_news", split="train[:500]")
texts  = ds["text"]
labels = [LABEL_NAMES[l] for l in ds["label"]]

print(f"Loaded {len(texts)} articles")
print(f"Category distribution: { {k: labels.count(k) for k in LABEL_NAMES.values()} }")
print(f"\nSample: {texts[0][:120]}")

## Create Haystack Documents

Each `Document` carries the article text as `content` and three metadata fields we can filter on:
the news `category` (string), the article's position `pos` (int) and its `length` in characters (int).

In [ ]:
from haystack import Document

haystack_docs = [
    Document(id=str(i), content=text, meta={"category": label, "pos": i, "length": len(text)})
    for i, (text, label) in enumerate(zip(texts, labels))
]

print(f"Created {len(haystack_docs)} Haystack documents")
print(f"Example: {haystack_docs[0]}")

## Embed and index documents

`SimlarDocumentStore` does not own an embedder: documents must arrive **with embeddings already set**.
`SentenceTransformersDocumentEmbedder` sets the `embedding` field on each document, and we then write them to the store.
Writes go straight to the index — writing more documents later appends only those documents.

In [ ]:
from haystack_integrations.components.embedders.sentence_transformers import SentenceTransformersDocumentEmbedder
from simlar.integrations.haystack.simlar_document_store import SimlarDocumentStore

MODEL = "sentence-transformers/all-MiniLM-L6-v2"

doc_embedder = SentenceTransformersDocumentEmbedder(model=MODEL)
doc_embedder.warm_up()

embedded_docs = doc_embedder.run(documents=haystack_docs)["documents"]
print(f"Embedded {len(embedded_docs)} documents  |  dim: {len(embedded_docs[0].embedding)}")

store = SimlarDocumentStore()
n_written = store.write_documents(embedded_docs)
print(f"Indexed {n_written} documents  |  store total: {store.count_documents()}")

## Build the query pipeline

The pipeline has two components:
1. `SentenceTransformersTextEmbedder` — encodes the query string into a vector
2. `SimlarHybridRetriever` — takes the raw query text and the query embedding, runs BM25 + vector search fused with RRF, and returns ranked `Document` objects

We connect the embedder's output to the retriever's `query_embedding` input.
The query text, and any `filters`, go to the retriever as run-time inputs.

In [ ]:
from haystack import Pipeline
from haystack_integrations.components.embedders.sentence_transformers import SentenceTransformersTextEmbedder
from simlar.integrations.haystack.simlar_retriever import SimlarHybridRetriever

query_pipeline = Pipeline()
query_pipeline.add_component("text_embedder", SentenceTransformersTextEmbedder(model=MODEL))
query_pipeline.add_component("retriever", SimlarHybridRetriever(document_store=store, top_k=5))
query_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")

print("Query pipeline ready")

## Run queries

Pass the question to both `text_embedder.text` and `retriever.query`.
Each result carries its fused RRF score in `Document.score`; `meta` holds only the metadata you wrote.

In [ ]:
def show(docs):
    for doc in docs:
        print(f"  [{doc.meta['category']:8s}]  score={doc.score:.4f}  len={doc.meta['length']:3d}  {doc.content[:80]}")
    print()


def run_query(question, filters=None, roles=None):
    retriever_inputs = {"query": question}
    if filters is not None:
        retriever_inputs["filters"] = filters
    if roles is not None:
        retriever_inputs["roles"] = roles
    result = query_pipeline.run({"text_embedder": {"text": question}, "retriever": retriever_inputs})
    return result["retriever"]["documents"]


for question in [
    "technology startup funding Silicon Valley",
    "Olympic Games world record athlete",
    "interest rate Federal Reserve inflation",
]:
    print(f"Query: '{question}'")
    show(run_query(question))

## Filtered retrieval

Pass `filters` to the retriever at run time, using the standard [Haystack filter syntax](https://docs.haystack.deepset.ai/docs/metadata-filtering) — the same dicts other Haystack document stores accept.

| Kind | Operators |
|---|---|
| Comparison | `==`, `!=`, `>`, `>=`, `<`, `<=`, `in`, `not in` |
| Logical | `AND`, `OR`, `NOT`, nested to any depth |

Fields are `meta.<key>` for metadata, or `content` / `id` for document fields.
The semantics follow Haystack's reference implementation: a document missing a field compares as `None`, string values compare as ISO dates under the ordering operators, and an invalid filter raises `FilterError`.

The filter is resolved to the matching documents *before* ranking, so a filtered query still returns a full top-k from the filtered set, not the few matches that happen to land in an unfiltered top-k.

In [ ]:
question = "stock market earnings profit"

print("Unfiltered:")
show(run_query(question))

print("Business only:")
show(run_query(question, filters={"field": "meta.category", "operator": "==", "value": "Business"}))

print("Business or Sci/Tech, at least 250 characters long:")
show(run_query(
    question,
    filters={
        "operator": "AND",
        "conditions": [
            {"field": "meta.category", "operator": "in", "value": ["Business", "Sci/Tech"]},
            {"field": "meta.length", "operator": ">=", "value": 250},
        ],
    },
))

print("Anything but World or Business, from the first 250 articles:")
show(run_query(
    question,
    filters={
        "operator": "AND",
        "conditions": [
            {"operator": "NOT", "conditions": [
                {"field": "meta.category", "operator": "in", "value": ["World", "Business"]},
            ]},
            {"field": "meta.pos", "operator": "<", "value": 250},
        ],
    },
))

## Init-time filters and `filter_policy`

A retriever can also carry `filters` from construction, applied to every run. `filter_policy` decides how run-time filters combine with them:

| Policy | Effect |
|---|---|
| `REPLACE` (default) | Run-time filters replace the init filters |
| `MERGE` | Run-time and init filters are combined with `AND` |

Here a retriever is pinned to Sports, and each run can narrow it further.

In [ ]:
from haystack.document_stores.types import FilterPolicy

sports_retriever = SimlarHybridRetriever(
    document_store=store,
    top_k=3,
    filters={"field": "meta.category", "operator": "==", "value": "Sports"},
    filter_policy=FilterPolicy.MERGE,
)

question = "championship final win"
q_emb = SentenceTransformersTextEmbedder(model=MODEL).run(text=question)["embedding"]

print("Init filter only (Sports):")
show(sports_retriever.run(query=question, query_embedding=q_emb)["documents"])

print("Merged with a run-time filter (Sports AND length < 200):")
show(sports_retriever.run(
    query=question,
    query_embedding=q_emb,
    filters={"field": "meta.length", "operator": "<", "value": 200},
)["documents"])

## Embedding-only and BM25-only retrieval

`SimlarEmbeddingRetriever` (vector) and `SimlarBM25Retriever` (keyword) take the same `filters`, `filter_policy` and `roles` arguments.
The store exposes the same three searches directly as `embedding_retrieval`, `bm25_retrieval` and `hybrid_retrieval`.

In [ ]:
from simlar.integrations.haystack.simlar_retriever import SimlarBM25Retriever, SimlarEmbeddingRetriever

question = "oil prices"
q_emb = SentenceTransformersTextEmbedder(model=MODEL).run(text=question)["embedding"]
not_world = {"field": "meta.category", "operator": "!=", "value": "World"}

print("Embedding retriever, category != World:")
show(SimlarEmbeddingRetriever(document_store=store, top_k=3).run(query_embedding=q_emb, filters=not_world)["documents"])

print("BM25 retriever, category != World:")
show(SimlarBM25Retriever(document_store=store, top_k=3).run(query=question, filters=not_world)["documents"])

print("Hybrid, called on the store directly:")
show(store.hybrid_retrieval(question, q_emb, filters=not_world, top_k=3))

## Role-based access

Documents can be granted to roles — at write time with `write_documents(docs, roles=[[...], ...])`, or afterwards with `grant` / `revoke`.
Passing `roles` to a retrieval returns only documents granted to at least one of them; `roles=None` (the default) skips the check.
Roles combine with `filters`, and both are resolved before ranking.

In [ ]:
business_ids = [d.id for d in store.filter_documents({"field": "meta.category", "operator": "==", "value": "Business"})]
store.grant(business_ids, ["finance_desk"])

question = "new software release"

print("No roles — every document is eligible:")
show(run_query(question)[:3])

print("roles=['finance_desk'] — only Business documents were granted:")
show(run_query(question, roles=["finance_desk"])[:3])

print("roles=['finance_desk'] AND length > 300:")
show(run_query(question, roles=["finance_desk"], filters={"field": "meta.length", "operator": ">", "value": 300})[:3])

print(f"Visible to 'finance_desk': {len(store.filter_documents(roles=['finance_desk']))} documents")

## Inspect and count by filter

`filter_documents(filters)` returns every matching document in insertion order (no ranking), and `count_documents_by_filter(filters)` counts them.

In [ ]:
print(f"Total documents in store : {store.count_documents()}")
for cat in LABEL_NAMES.values():
    n = store.count_documents_by_filter({"field": "meta.category", "operator": "==", "value": cat})
    print(f"  {cat:10s}: {n}")

long_tech = store.filter_documents(
    {
        "operator": "AND",
        "conditions": [
            {"field": "meta.category", "operator": "==", "value": "Sci/Tech"},
            {"field": "meta.length", "operator": ">", "value": 400},
        ],
    }
)
print(f"\nSci/Tech articles longer than 400 characters: {len(long_tech)}")
for doc in long_tech[:3]:
    print(f"  pos={doc.meta['pos']:3d}  {doc.content[:90]}")

## Metadata introspection

These helpers let you explore what's in the store without loading every document:

| Method | Returns |
|---|---|
| `get_metadata_fields_info()` | Field names → inferred type (`keyword`, `int`, `float`, `boolean`) |
| `get_metadata_field_min_max(field)` | `{"min": …, "max": …}` for a numeric or sortable field |
| `get_metadata_field_unique_values(field, search_term, from_, size, filters)` | A page of distinct values, and the total count |
| `count_unique_metadata_by_filter(filters, fields)` | Unique-value count per field, scoped by a filter |

In [ ]:
print("Metadata schema:")
for field, info in store.get_metadata_fields_info().items():
    print(f"  {field:10s}: {info['type']}")

length_range = store.get_metadata_field_min_max("meta.length")
print(f"\nlength range : {length_range['min']} – {length_range['max']}")

categories, total = store.get_metadata_field_unique_values("meta.category")
print(f"Unique categories ({total}) : {categories}")

# Which categories have articles shorter than 150 characters?
short_cats, _ = store.get_metadata_field_unique_values(
    "meta.category", filters={"field": "meta.length", "operator": "<", "value": 150}
)
print(f"Categories with articles < 150 chars : {short_cats}")

counts = store.count_unique_metadata_by_filter(
    filters={"field": "meta.category", "operator": "in", "value": ["Sports", "World"]},
    metadata_fields=["meta.category", "meta.pos"],
)
print(f"Unique values in the Sports+World slice : {counts}")

## Update metadata by filter

`update_by_filter(filters, meta)` merges `meta` into every matching document. New fields are filterable straight away.

In [ ]:
n_updated = store.update_by_filter(
    filters={"field": "meta.category", "operator": "==", "value": "Sci/Tech"},
    meta={"reviewed": True},
)
print(f"Tagged {n_updated} Sci/Tech documents as reviewed\n")

print("Retrieval restricted to reviewed documents:")
show(run_query("space exploration", filters={"field": "meta.reviewed", "operator": "==", "value": True})[:3])

# Documents without the field compare as None, so != True matches them
unreviewed = store.count_documents_by_filter({"field": "meta.reviewed", "operator": "!=", "value": True})
print(f"Documents not reviewed : {unreviewed}")

## Delete documents

Deletes go straight to the index and compact it — no tombstones, no rebuild.

| Method | Use when |
|---|---|
| `delete_documents(ids)` | You have exact document IDs |
| `delete_by_filter(filters)` | You want to remove everything matching a filter |
| `delete_all_documents()` | Full reset |

In [ ]:
print(f"Before deletion : {store.count_documents()} documents")

store.delete_documents(["0"])
print(f"After deleting id '0' : {store.count_documents()} documents")

n_deleted = store.delete_by_filter({"field": "meta.category", "operator": "==", "value": "World"})
print(f"After deleting World articles ({n_deleted} removed) : {store.count_documents()} documents")

print("\nRemaining category breakdown:")
for cat in LABEL_NAMES.values():
    n = store.count_documents_by_filter({"field": "meta.category", "operator": "==", "value": cat})
    print(f"  {cat:10s}: {n}")

print("\nWorld no longer appears, even unfiltered:")
show(run_query("government elections war")[:3])

## Persistence: save and reload

`save(path)` writes the index (including metadata and role grants) and the documents to a directory;
`SimlarDocumentStore.load(path)` reconstructs the store without re-embedding.

The retrievers and the store also serialize with `to_dict` / `from_dict`, so a whole pipeline round-trips through `Pipeline.dumps()` / `Pipeline.loads()`.
Note that `to_dict` saves the store's settings, not its documents — load the documents with `SimlarDocumentStore.load` and attach that store to the loaded pipeline.

In [ ]:
import tempfile
from pathlib import Path

question = "company shares fall"
q_emb = SentenceTransformersTextEmbedder(model=MODEL).run(text=question)["embedding"]
filters = {"field": "meta.length", "operator": ">=", "value": 200}

with tempfile.TemporaryDirectory() as tmp:
    save_path = Path(tmp) / "ag_news_store"
    store.save(save_path)
    print(f"Files written: {sorted(f.name for f in save_path.iterdir())}")

    reloaded_store = SimlarDocumentStore.load(save_path)
    print(f"Reloaded store: {reloaded_store.count_documents()} documents")

    original = [d.id for d in store.hybrid_retrieval(question, q_emb, filters=filters, roles=["finance_desk"])]
    restored = [d.id for d in reloaded_store.hybrid_retrieval(question, q_emb, filters=filters, roles=["finance_desk"])]
    print(f"Filtered + role-scoped results match after reload: {original == restored}")

# Serialize the pipeline definition
yaml_str = query_pipeline.dumps()
loaded_pipeline = Pipeline.loads(yaml_str)
print(f"\nPipeline round-trip components: {list(loaded_pipeline.to_dict()['components'])}")